# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a complete workflow for loading and exploring the FAIRⁱ2 (FAIR^2) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The data contains mass spectrometry protein-level records with annotation from human samples, including protein abundance, peptide counts, molecular weights, and additional features.

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is available via URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading

We will load the Croissant metadata and enumerate the main dataset attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # this is an mlcroissant.DatasetMetadata object

print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")


## 2. Data Overview

Let's enumerate all available record sets (tables), the fields (columns), and their `@id` identifiers.

`mlcroissant` exposes these via the Dataset object. We reference everything by their `@id` as per Croissant best practice.

In [ ]:
# Print all available record sets and their fields with `@id`s
from pprint import pprint

record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1} @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    print(f"  Description: {rs.get('description', '(no description)')}")
    # List the fields/columns
    fields = rs.get('field', [])
    print("  Fields/Columns:")
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"    - @id: {field['@id']}, name: {field.get('name', '(no name)')}, type: {field.get('dataType', '(unknown)')}")
    print()


## 3. Data Extraction

Let's load all available record sets into pandas DataFrames for further analysis.

First, we collect the `@id`s of the record sets, then extract their records into DataFrames, mapping the content by their Croissant `@id`.

In [ ]:
# Get all record set @ids
record_set_ids = [rs["@id"] for rs in record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records. Columns (by @id): {list(df.columns)}\n")
        else:
            print("No records found for this record set.\n")
    except Exception as e:
        print(f"Error loading records: {e}\n")

# Display the head of the first (main) record set
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Displaying first few records of main record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes extracted!")

## 4. Exploratory Data Analysis (EDA)

Let us process and explore the main record set. For demonstration, we'll select a numeric field (e.g., `MW` for molecular weight or a peptide count field) using its `@id`.

We will:
- Filter records above a threshold,
- Normalize the field,
- Optionally, group by a categorical field if present,
- All references use the schema `@id` for traceability.

In [ ]:
# For this dataset, common numeric fields are "MW" (Molecular Weight), peptide counts, etc.
# Let's attempt to auto-detect numeric fields or use MW if available.

import numpy as np

df = dataframes[main_rs_id]
numeric_field_ids = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
if not numeric_field_ids:
    # try string fields that look like numbers
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    print(f"Selected numeric field for EDA: {numeric_field}")
else:
    print("No numeric field detected.")
    numeric_field = None

if numeric_field is not None:
    # Drop missing values for this analysis
    filtered_df = df[df[numeric_field] > 0].copy()
    print(f"Filtered records with {numeric_field} > 0. Count: {len(filtered_df)}")

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a label/description/id field
    group_field_candidates = [
        col for col in df.columns if 'label' in col.lower() or 'desc' in col.lower() or 'id' in col.lower()
    ]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        print(f"Grouping records by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numerical field found for EDA.")


## 5. Visualization

Let's visualize the distribution of the selected numeric field and, if grouped, show grouped means.

_Note: Visualization will only run if a numeric field is present and Matplotlib/Seaborn available._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(10,4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(12,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No numeric field found to visualize.")

## 6. Conclusion

In this notebook, we've used the `mlcroissant` Python library to inspect and analyze the FAIRⁱ2 mass spectrometry dataset described by the Croissant schema. 

- Data structure (record sets, field `@id`s) was explored programmatically per the schema
- Data was extracted and analyzed using consistent `@id` references
- Example exploratory analysis and normalization were performed on numeric fields
- We demonstrated visualization for data distribution and basic grouping

For further analysis, tailor the EDA and visualizations to domain-specific fields (such as protein abundance and annotations) by referencing the Croissant `@id`s, as demonstrated.